# Wadi Restoration — Hajar Mountain Ecosystem Recovery

**Scenario:** A 5-hectare degraded wadi site in the Hajar Mountains near Ras Al Khaimah. The site was historically used for grazing and has lost much of its native vegetation cover. The goal is ecological restoration using only native species.

**Climate:** Arid desert hot (Köppen BWh). Seasonal flash flood potential. Extreme summer temperatures exceeding 48°C.

**Soil:** Rocky, well-drained wadi gravel with low organic matter. Non-saline.

**Design brief:** Restore native plant community structure. Maximize biodiversity using only indigenous species. Zero irrigation after establishment. Improve soil stability and create wildlife habitat.

---

## 1. Define the Wadi Site

In [ ]:
from natureos.site import Site, SoilProfile, ClimateZone, LandUse, SoilTexture

wadi_site = Site(
    name="Wadi Bih Restoration Site",
    climate_zone=ClimateZone.BWH,
    soil=SoilProfile(
        texture=SoilTexture.SAND,
        salinity_dsm=0.8,
        organic_matter_pct=0.3,
        ph=7.5,
        depth_cm=60.0
    ),
    area_hectares=5.0,
    land_use=LandUse.ECOLOGICAL_RESTORATION,
    annual_rainfall_mm=120.0,
    max_summer_temp_c=48.0,
    latitude=25.78,
    longitude=56.05
)

print(f"Site: {wadi_site.name}")
print(f"Area: {wadi_site.area_hectares} ha")
print(f"Arid: {wadi_site.is_arid}, Saline: {wadi_site.is_saline}")
print(f"Max temp: {wadi_site.max_summer_temp_c}°C")

## 2. Filter Native Mountain Wadi Species Only

In [ ]:
from natureos.data.mena_species import ALL_SPECIES
from natureos.species import EcosystemType

# Only species native to mountain wadi and desert scrub ecosystems
native_wadi_species = [
    sp for sp in ALL_SPECIES
    if sp.is_native and (
        EcosystemType.MOUNTAIN_WADI in sp.ecosystems or
        EcosystemType.DESERT_SCRUB in sp.ecosystems
    )
]

print(f"Native mountain/desert species: {len(native_wadi_species)}")
for sp in native_wadi_species:
    print(f"  • {sp.display_name} — {sp.growth_form.value}")

## 3. Habitat Suitability for Wadi Conditions

In [ ]:
from natureos.engines.habitat import HabitatSuitability

suitability = HabitatSuitability(wadi_site)
results = suitability.evaluate_all(native_wadi_species)

print("Habitat suitability for wadi restoration:\n")
for i, r in enumerate(results, 1):
    print(f"{i}. {r.summary()}")

## 4. Water Budget — Restoration Requires Zero Irrigation

In [ ]:
from natureos.engines.water import WaterBudget

# Use all suitable native species
restoration_species = [r.species for r in results]

water = WaterBudget(site=wadi_site, irrigation_efficiency=0.85)
water_result = water.calculate(restoration_species)

print(water_result.summary())
print(f"\nWith {wadi_site.annual_rainfall_mm}mm annual rainfall, "
      f"these native species should survive on rainfall alone after establishment.")

## 5. Biodiversity — Restoring Community Structure

In [ ]:
from natureos.engines.biodiversity import BiodiversityIndex

# Naturalistic planting densities for restoration
restoration_plan = {
    restoration_species[0]: 120,  # Dominant tree — scattered
    restoration_species[1]: 80,   # Secondary tree
    restoration_species[2]: 200,  # Shrub — matrix planting
}

# Add remaining shrubs at lower densities
for sp in restoration_species[3:]:
    restoration_plan[sp] = 150

bio = BiodiversityIndex(species_abundances=restoration_plan)
bio_result = bio.calculate()

print(bio_result.summary())
print(f"\nInterpretation: {bio_result.interpretation}")
print(f"Native ratio: {bio_result.native_ratio:.0%} — 100% native restoration")

## 6. Species Interactions — Natural Community Dynamics

In [ ]:
from natureos.engines.interactions import SpeciesInteraction

interaction = SpeciesInteraction(species_list=restoration_species)
interaction_result = interaction.analyse()

print(interaction_result.summary())
print()

if interaction_result.facilitation_pairs:
    print("🤝 Ecological facilitation detected:")
    for pair in interaction_result.facilitation_pairs:
        print(f"  • {pair.summary()}")
        print(f"    {pair.rationale}")

if interaction_result.conflict_pairs:
    print("\n⚠️  Potential conflicts:")
    for pair in interaction_result.conflict_pairs:
        print(f"  • {pair.summary()}")
else:
    print("\n✅ No significant conflicts — species co-occur naturally in wadi ecosystems.")

## 7. Carbon Sequestration — Long-Term Climate Benefit

In [ ]:
from natureos.engines.carbon import CarbonEstimator

carbon = CarbonEstimator(
    species_counts=restoration_plan,
    site_area_hectares=wadi_site.area_hectares,
    ecosystem_type="desert_scrub",
    time_horizon_years=30
)
carbon_result = carbon.calculate()

print(carbon_result.summary())
print(f"\nOver 30 years, this restored wadi will sequester "
      f"approximately {carbon_result.co2_equivalent_t:.1f} tonnes of CO₂ equivalent "
      f"while restoring native habitat.")

---

## Summary: Wadi Restoration

This notebook demonstrates NatureOS for **ecological restoration** — a fundamentally different use case from urban design:

- **100% native species** — no introduced or naturalized species
- **Zero irrigation target** — species must survive on rainfall alone
- **Natural community structure** — species that co-occur in the wild
- **Long time horizon** — 30-year carbon and ecosystem recovery

The same engines used for a Dubai park work equally well for mountain ecosystem restoration. Only the site parameters, species pool, and objectives change — the computational infrastructure remains the same.

---

*NatureOS Core v0.1.0 | MENA Species Dataset v0.1.0 | Apache 2.0 License*